# MCMC Island Hopping: The Metropolis-Hastings Algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/metropolis_island_hopping.ipynb)

**Companion blog post:** [MCMC Island Hopping: An Intuitive Guide to the Metropolis-Hastings Algorithm](https://sesen.ai/blog/mcmc-metropolis-hastings-island-hopping-guide)

---

In this notebook, you'll implement the Metropolis-Hastings algorithm from scratch using the classic "island hopping" analogy.

By the end, you'll understand:
- How a simple random walk can recover any target distribution
- Why the acceptance ratio works (and when to correct for asymmetric proposals)
- The role of burn-in and chain length
- How to diagnose convergence with trace plots

## 1. The Problem

Imagine you're a politician touring a chain of 7 islands. Each island has a different population:

| Island | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|--------|---|---|---|---|---|---|---|
| Population | 2 | 3 | 1 | 5 | 8 | 2 | 9 |

You want to **spend time on each island in proportion to its population** — more time on crowded islands, less on quiet ones.

The catch: you have no map. You can only see the island you're on and the one next door. You can ask the current island and a neighbouring island for their population, then decide whether to move.

This is exactly the problem the Metropolis-Hastings algorithm solves. In real applications, the "islands" are parameter values and the "populations" are posterior probabilities.

## 2. Quick Win: Run the Complete Algorithm

Let's run the island hopper and see that it recovers the target distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def metropolis_island_hopping(target, n_samples, start=0, burn_in=1000):
    """
    Metropolis-Hastings sampler for a discrete distribution using
    nearest-neighbour proposals (the 'island hopping' variant).

    Args:
        target: List of unnormalised target values (e.g. island populations)
        n_samples: Total number of steps (including burn-in)
        start: Starting island index
        burn_in: Steps to discard before collecting samples

    Returns:
        visit_counts: Array of visit counts for each island
        trace: Full chain history (for diagnostics)
    """
    n_islands = len(target)
    visit_counts = np.zeros(n_islands)
    trace = []
    current = start

    for step in range(n_samples):
        # PROPOSE: pick a neighbour uniformly at random
        if current == 0:
            proposed = 1
        elif current == n_islands - 1:
            proposed = n_islands - 2
        else:
            proposed = current + np.random.choice([-1, 1])

        # CORRECTION: account for asymmetric proposals at boundaries
        n_neighbours_current = 2 if 0 < current < n_islands - 1 else 1
        n_neighbours_proposed = 2 if 0 < proposed < n_islands - 1 else 1
        proposal_ratio = n_neighbours_current / n_neighbours_proposed

        # ACCEPT/REJECT: Metropolis-Hastings ratio
        acceptance_ratio = (target[proposed] / target[current]) * proposal_ratio

        if acceptance_ratio >= 1 or np.random.random() < acceptance_ratio:
            current = proposed

        trace.append(current)

        # Record visit (after burn-in)
        if step >= burn_in:
            visit_counts[current] += 1

    return visit_counts, trace


# Target distribution: island populations
target = [2, 3, 1, 5, 8, 2, 9]

# Run the sampler
np.random.seed(42)
visits, trace = metropolis_island_hopping(target, n_samples=50_000, burn_in=1000)

# Compare: normalised target vs sampled distribution
target_normalised = np.array(target) / sum(target)
visits_normalised = visits / visits.sum()

islands = range(len(target))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - width/2 for i in islands], target_normalised, width, label='Target', alpha=0.8)
ax.bar([i + width/2 for i in islands], visits_normalised, width, label='MCMC samples', alpha=0.8)
ax.set_xlabel('Island')
ax.set_ylabel('Proportion of time')
ax.set_title('Metropolis-Hastings: Target vs Sampled Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print('Target (normalised):', [f'{p:.3f}' for p in target_normalised])
print('MCMC (normalised):  ', [f'{p:.3f}' for p in visits_normalised])

**You just ran MCMC!** The blue (target) and orange (sampled) bars should nearly overlap — the sampler spends time on each island in proportion to its population, even though it only ever looks at one neighbour at a time.

Now let's understand how it works.

## 3. What Just Happened?

The algorithm repeats a simple loop: **propose, evaluate, accept or reject**.

### The Proposal
At each step, the sampler looks at one of its neighbours (left or right) and considers moving there. At the boundaries, there's only one neighbour.

### The Acceptance Decision
If the proposed island has a larger population, always move. If smaller, move with probability equal to the ratio of populations.

For example, from island 4 (population 8) proposing island 3 (population 5):
- Acceptance ratio = 5/8 = 0.625
- Move with 62.5% probability

This means the chain sometimes visits less popular islands — essential for exploring the full distribution.

### The Hastings Correction
At boundary islands (0 and 6), the proposal distribution is asymmetric: from island 0 you always propose island 1, but from island 1 you only propose island 0 half the time. The correction factor accounts for this.

Let's trace through the first few steps:

In [ ]:
# Trace through the first 20 steps manually
np.random.seed(42)
target = [2, 3, 1, 5, 8, 2, 9]
current = 0
n_islands = len(target)

print(f"Starting on island {current} (population {target[current]})")
print("=" * 70)

for step in range(20):
    # Propose
    if current == 0:
        proposed = 1
    elif current == n_islands - 1:
        proposed = n_islands - 2
    else:
        proposed = current + np.random.choice([-1, 1])

    # Correction
    n_curr = 2 if 0 < current < n_islands - 1 else 1
    n_prop = 2 if 0 < proposed < n_islands - 1 else 1
    proposal_ratio = n_curr / n_prop

    # Accept/reject
    ratio = (target[proposed] / target[current]) * proposal_ratio
    u = np.random.random()
    accepted = ratio >= 1 or u < ratio

    correction_str = f" x {proposal_ratio:.1f}" if proposal_ratio != 1.0 else ""
    print(f"Step {step+1:2d}: island {current} (pop {target[current]}) -> "
          f"propose {proposed} (pop {target[proposed]}), "
          f"ratio = {target[proposed]}/{target[current]}{correction_str} = {ratio:.2f}, "
          f"u = {u:.2f} -> {'ACCEPT' if accepted else 'REJECT'}")

    if accepted:
        current = proposed

## 4. Diagnostics: Is My Chain Working?

### Trace Plot
A trace plot shows which island the chain visits at each step. A well-mixed chain should look like random noise — visiting all islands without getting stuck.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Full trace
axes[0].plot(trace[:5000], linewidth=0.5, alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Island')
axes[0].set_title('Trace Plot (first 5,000 steps)')
axes[0].axvline(x=1000, color='r', linestyle='--', label='End of burn-in')
axes[0].legend()

# Zoomed in
axes[1].plot(trace[:200], 'o-', markersize=2, linewidth=0.8)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Island')
axes[1].set_title('Trace Plot (first 200 steps — zoomed in)')

plt.tight_layout()
plt.show()

### Cumulative Distribution Convergence

Let's watch the sampled distribution converge to the target as we add more samples:

In [ ]:
target = [2, 3, 1, 5, 8, 2, 9]
target_norm = np.array(target) / sum(target)
burn_in = 1000

checkpoints = [1_000, 5_000, 10_000, 50_000]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, n in zip(axes, checkpoints):
    # Count visits after burn-in
    post_burnin = [t for i, t in enumerate(trace) if i >= burn_in and i < n]
    counts = np.bincount(post_burnin, minlength=len(target))
    counts_norm = counts / counts.sum() if counts.sum() > 0 else counts

    width = 0.35
    islands = range(len(target))
    ax.bar([i - width/2 for i in islands], target_norm, width, label='Target', alpha=0.8)
    ax.bar([i + width/2 for i in islands], counts_norm, width, label='MCMC', alpha=0.8)
    ax.set_title(f'n = {n:,}')
    ax.set_xlabel('Island')
    ax.set_ylim(0, 0.4)
    if ax == axes[0]:
        ax.legend(fontsize=8)

plt.suptitle('Convergence: More Samples = Better Approximation', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Going Deeper: The Mathematics

### The General Metropolis-Hastings Algorithm

Given a target distribution $\pi(x)$ (known up to a normalising constant) and a proposal distribution $q(x' | x)$:

1. Start at some state $x_0$
2. At each step $t$:
   - Propose $x'$ from $q(x' | x_t)$
   - Compute acceptance probability: $\alpha = \min\left(1, \frac{\pi(x') \, q(x_t | x')}{\pi(x_t) \, q(x' | x_t)}\right)$
   - With probability $\alpha$, set $x_{t+1} = x'$; otherwise $x_{t+1} = x_t$

### Why It Works: Detailed Balance

The algorithm satisfies **detailed balance**:

$$\pi(x) \, T(x' | x) = \pi(x') \, T(x | x')$$

where $T$ is the transition kernel. This means the chain has $\pi$ as its stationary distribution — running it long enough guarantees convergence.

### In Our Island Example

- $\pi(x)$ = island populations (unnormalised)
- $q(x' | x)$ = uniform over neighbours
- The Hastings correction handles the asymmetry at boundaries

## 6. Exercises

Try these modifications to deepen your understanding.

In [ ]:
# Exercise 1: Cyclic islands
# Modify the algorithm so that island 0 and island 6 are neighbours
# (the chain wraps around). Does this improve mixing?

# Hint: change the proposal step so boundary islands have 2 neighbours
# and the correction factor is always 1.

# Your code here:
# def metropolis_cyclic(target, n_samples, start=0, burn_in=1000):
#     ...

In [ ]:
# Exercise 2: Burn-in sensitivity
# Run the sampler with burn_in = 0, 100, 1000, 5000
# and compare the resulting distributions.
# How does the starting island affect the result with no burn-in?

burn_in_values = [0, 100, 1000, 5000]
target = [2, 3, 1, 5, 8, 2, 9]

# Your code here:
# for bi in burn_in_values:
#     visits, _ = metropolis_island_hopping(target, 50_000, start=0, burn_in=bi)
#     ...

In [ ]:
# Exercise 3: Continuous target distribution
# Replace the discrete islands with a continuous Gaussian target.
# Use a Gaussian proposal: x' ~ N(x_current, sigma^2)

# Hint: 
# - target(x) = exp(-x^2 / 2) (standard normal, unnormalised)
# - proposal: x' = x + np.random.normal(0, sigma)
# - Since the proposal is symmetric, the correction factor is 1

# Your code here:
# def metropolis_continuous(target_fn, n_samples, sigma=1.0, start=0.0, burn_in=1000):
#     ...

In [ ]:
# Exercise 4: Proposal width tuning
# For the continuous case, try different proposal widths (sigma).
# Plot the acceptance rate vs sigma.
# What is the "sweet spot"? (Aim for ~23% acceptance in high dimensions,
# ~44% in 1D — see Roberts et al. 1997)

# Your code here:
# sigmas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
# for sigma in sigmas:
#     ...

In [ ]:
# Exercise 5: Multiple chains
# Run 4 chains with different starting positions and overlay their trace plots.
# Do they all converge to the same distribution? How long does it take?

target = [2, 3, 1, 5, 8, 2, 9]
starts = [0, 2, 4, 6]

# Your code here:
# fig, ax = plt.subplots(figsize=(12, 4))
# for s in starts:
#     _, trace = metropolis_island_hopping(target, 5000, start=s, burn_in=0)
#     ax.plot(trace, alpha=0.6, label=f'start={s}')
# ...

## 7. The Foundations

### Historical Context

The Metropolis algorithm was developed at **Los Alamos National Laboratory** in 1953 by Nicholas Metropolis, Arianna and Marshall Rosenbluth, and Augusta and Edward Teller. They needed to simulate molecular behaviour to compute equations of state — a problem that required integrating over astronomically many configurations.

Their key insight: instead of evaluating an intractable integral, **generate representative samples** using a carefully designed random walk.

In 1970, W.K. Hastings generalised the algorithm to allow **asymmetric proposal distributions**, vastly expanding its applicability.

### Key Papers

- **Metropolis, N., Rosenbluth, A.W., Rosenbluth, M.N., Teller, A.H. and Teller, E.** (1953). ["Equation of State Calculations by Fast Computing Machines"](https://bayes.wustl.edu/Manual/EquationsOfState.pdf). *The Journal of Chemical Physics*, 21(6), pp.1087–1092.

- **Hastings, W.K.** (1970). ["Monte Carlo Sampling Methods Using Markov Chains and Their Applications"](https://academic.oup.com/biomet/article-abstract/57/1/97/284580). *Biometrika*, 57(1), pp.97–109.

- **Chib, S. and Greenberg, E.** (1995). "Understanding the Metropolis-Hastings Algorithm". *The American Statistician*, 49(4), pp.327–335.

### Further Reading

- **Bishop's PRML Chapter 11** — Modern textbook coverage of MCMC methods
- **Stan documentation** — See HMC/NUTS in action for practical Bayesian inference
- **Next: Gibbs Sampling** — A special case of MH where proposals are always accepted

---

**Author:** Dr. Berkan Sesen | [sesen.ai](https://sesen.ai)

**Companion blog post:** [MCMC Island Hopping: An Intuitive Guide to the Metropolis-Hastings Algorithm](https://sesen.ai/blog/mcmc-metropolis-hastings-island-hopping-guide)